In [ ]:
import os

from plot_benchmark import load_data, convert_data, plot_training_and_gpu_hours

In [ ]:
output_folder = "plots_parameters"
input_folder = "benchmark"

data = load_data(input_folder)
df = convert_data(data)
os.makedirs(output_folder, exist_ok=True)

In [ ]:
plot_training_and_gpu_hours(
    df, plot_name=None, plot_title="", output_folder=output_folder
)

In [ ]:
tp_df = df[
    (df["arch"] == "llama8b")
    & (df["pp"] == 1)
    & (df["precision"] == "bf16")  # noqa E712
    & (df["seq_length"] == 4096)
    & (df["cp"] == 1)
]
tp_df = tp_df.sort_values(by=["tp", "batch_size"])
plot_training_and_gpu_hours(
    tp_df,
    plot_name=None,
    plot_title="Impact of tensor parallelism on training",
    output_folder=output_folder,
)

In [ ]:

seq_df = df[
    (df["arch"] == "llama8b") & (df["tp"] == 1) & (df["precision"] == "bf16")
]  # noqa E712
seq_df = seq_df.sort_values(by=["seq_length", "batch_size", "pp", "cp"])
plot_training_and_gpu_hours(
    seq_df,
    plot_name=None,
    plot_title="Impact of Seq Length on training",
    output_folder=output_folder,
)


In [ ]:

precision_batch_df = df[
    (df["pp"] == 1) & (df["tp"] == 1) & (df["arch"] == "llama8b")
]
precision_batch_df = precision_batch_df.sort_values(by=["precision", "batch_size"])

plot_training_and_gpu_hours(
    precision_batch_df,
    plot_name=None,
    plot_title="Impact of Precision and Batch_size on training",
    output_folder=output_folder,
)


In [ ]:

reduced_df = df[
    (
        (df["arch"] == "llama1b")
        & (df["precision"] == "bf16")
        & (df["batch_size"] == 1024)
    )
    | (
        (df["arch"] == "llama3b")
        & (df["precision"] == "fp8")
        & (df["batch_size"] == 1024)
    )
    | (
        (df["pp"] == 1)
        & (df["tp"] == 1)
        & (df["cp"] == 1)
        & (df["arch"] == "llama8b")
        & (df["batch_size"] == 1024)
    )
    | (
        (df["pp"] == 4)
        & (df["tp"] == 4)
        & (df["cp"] == 2)
        & (df["arch"] == "llama70b")
        & (df["batch_size"] == 512)
    )
    | (
        (df["arch"] == "nemotronh8b")
        # & (df["precision"] == "fp8")
        & (df["tp"] == 1)
    )
    | ((df["arch"] == "mambahybrid8b") & (df["tp"] == 1))
    | ((df["precision"] == "bf16") & (df["arch"] == "mixtral7x8"))
]

archs = [
    "llama1b",
    "llama3b",
    ["llama1b", "llama3b"],
    ["llama1b", "llama3b", "llama8b"],
    "llama8b",
    "llama70b",
    "mambahybrid8b",
    "mixtral8x7",
    "nemotronh8b",
    ["nemotronh8b", "mambahybrid8b"],
    ["llama1b", "llama3b", "llama8b", "nemotronh8b"],
]
archs.append([a for a in archs if isinstance(a, str)])
os.makedirs(os.path.join(output_folder, "archs"), exist_ok=True)
for arch in archs:
    plot_title = "Impact of Architecture and FP8 on training"
    if isinstance(arch, list):
        arch_df = reduced_df[(reduced_df["arch"].isin(arch))]
        arch = "_".join(arch)
    else:
        arch_df = df[(df["arch"] == arch)]
        plot_title = f"Impact of parameters for training a {arch}"
    arch_df = arch_df.sort_values(by=["arch", "precision"])
    plot_training_and_gpu_hours(
        arch_df,
        plot_name=None,
        plot_title=plot_title,
        output_folder=os.path.join(output_folder, "archs"),
    )